# EDA de las predicciones de VLMs sobre Multimodal3DIdent (split *test*)

**Objetivo.** Caracterizar las descripciones generadas por cada combinación *modelo × prompt* antes de calcular métricas finales:
qué tan largas y regulares son, qué vocabulario usan y, sobre todo, **qué atributos generativos de la escena recuperan correctamente**.

Cada imagen $i$ del conjunto de prueba está generada por un vector de factores conocidos
$$
z_i = \big(s_i,\; x_i,\; y_i,\; h^{\text{obj}}_i,\; h^{\text{spot}}_i,\; h^{\text{bg}}_i\big),
$$
con $s_i \in \{0,\dots,6\}$ la forma del objeto, $(x_i, y_i) \in \{0,1,2\}^2$ la posición discretizada y
$h^{(\cdot)}_i \in [0,1)$ los tonos (*hue*) del objeto, del foco y del fondo. Para cada corrida $r$ (modelo, prompt) el VLM produce un texto $\hat t_{i,r}$,
del que extraemos heurísticamente una estimación $\hat z_{i,r}$ atributo por atributo.

**Estructura**

1. Configuración y carga de corridas desde `runs/` (modelo y prompt se obtienen parseando `exp_id`).
2. Controles de integridad: cobertura del manifest, duplicados, predicciones vacías.
3. Distribución de los factores verdaderos en el split de prueba.
4. Léxicos y validación de los extractores contra el texto de referencia.
5. Extracción de atributos desde las predicciones.
6. Longitud, truncamiento y degeneración del texto.
7. Vocabulario.
8. Fidelidad por atributo (con intervalos de Wilson y línea base mayoritaria).
9. Análisis de errores: forma, posición, color, contraste objeto–fondo.
10. Comparaciones pareadas (McNemar exacto con corrección de Holm).
11. Dificultad por imagen y ejemplos cualitativos.
12. Exportación de tablas.

> **Advertencia metodológica.** La extracción de atributos es léxica (expresiones regulares), no semántica.
> Sirve para un EDA y para detectar patrones de error, pero sus tasas son **cotas aproximadas**: pueden existir falsos positivos
> (p. ej. *"head"* en *"the dragon's head"*) y falsos negativos (sinónimos no incluidos). La sección 4 cuantifica la precisión
> de los extractores sobre el texto de referencia, cuyo contenido es conocido.

## 1. Configuración

In [ ]:
import colorsys
import itertools
import re
import sys
import warnings
from collections import Counter
from pathlib import Path

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy import stats

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RUNS_DIR = ROOT / "runs"
MANIFEST = ROOT / "data/manifests/m3di_test.parquet"
FIG_DIR = ROOT / "reports/figures/eval"
TAB_DIR = ROOT / "reports/tables/eval"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

# ── Parámetros del análisis ──────────────────────────────────────────────────
SEED = 7006
ALPHA = 0.05  # nivel para IC de Wilson y pruebas
KEEP = {
    "dataset": "m3di",
    "split": "test",
}  # filtra corridas por lo parseado en exp_id (None = no filtrar)
MODEL_LABELS = {}  # opcional, p. ej. {'internvl': 'InternVL3.5-8B', 'qwen25vl': 'Qwen2.5-VL-7B'}
PROMPT_LABELS = {}  # opcional, p. ej. {'p0_minimal': 'P0 · mínimo'}
N_EXAMPLES = 3  # ejemplos cualitativos por corrida

RNG = np.random.default_rng(SEED)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 160)
warnings.filterwarnings("ignore", category=FutureWarning)

ModuleNotFoundError: No module named 'scipy'

In [ ]:
# ── Estilo de figuras ────────────────────────────────────────────────────────
# Paleta categórica en orden fijo (color = identidad del modelo, nunca su rango)
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
INK, INK2, INK3, GRID, SURF = "#0b0b0b", "#52514e", "#8a8984", "#e6e5e1", "#fcfcfb"
# Secuencial (magnitud) de un solo tono y divergente con punto medio gris
SEQ = mcolors.LinearSegmentedColormap.from_list(
    "seq_blue",
    ["#f4f8fe", "#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"],
)
DIV = mcolors.LinearSegmentedColormap.from_list(
    "div_blue_red", ["#184f95", "#6da7ec", "#f0efec", "#ee9190", "#c23a39"]
)

plt.rcParams.update(
    {
        "figure.facecolor": SURF,
        "axes.facecolor": SURF,
        "savefig.facecolor": SURF,
        "axes.edgecolor": INK3,
        "axes.labelcolor": INK2,
        "axes.titlecolor": INK,
        "axes.titlesize": 11,
        "axes.titleweight": "semibold",
        "axes.labelsize": 9.5,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.color": GRID,
        "grid.linewidth": 0.8,
        "axes.axisbelow": True,
        "xtick.color": INK2,
        "ytick.color": INK2,
        "xtick.labelsize": 8.5,
        "ytick.labelsize": 8.5,
        "legend.frameon": False,
        "legend.fontsize": 8.5,
        "font.size": 9.5,
        "lines.linewidth": 2,
        "lines.markersize": 6,
        "figure.dpi": 110,
    }
)


def savefig(fig, name):
    # Guarda PNG (revisión) y PDF (informe LaTeX).
    for ext in ("png", "pdf"):
        fig.savefig(FIG_DIR / f"{name}.{ext}", dpi=200, bbox_inches="tight")


def save_table(df, name, index=True, float_format="%.3f"):
    # Guarda CSV y, si es posible, .tex (booktabs) para el informe.
    df.to_csv(TAB_DIR / f"{name}.csv", index=index)
    try:
        df.to_latex(
            TAB_DIR / f"{name}.tex",
            index=index,
            float_format=float_format,
            escape=True,
            na_rep="--",
        )
    except Exception as e:  # jinja2 ausente u otro problema de formato
        print(f"[aviso] {name}.tex no generado: {e}")


def annotate_heatmap(ax, M, fmt="{:.2f}", thresh=0.55, vmin=0, vmax=1):
    # Escribe el valor de cada celda con tinta que contraste con el relleno.
    ax.tick_params(length=0)
    for (r, c), v in np.ndenumerate(M):
        if np.isnan(v):
            continue
        rel = (v - vmin) / (vmax - vmin + 1e-12)
        ax.text(
            c + 0.5,
            r + 0.5,
            fmt.format(v),
            ha="center",
            va="center",
            fontsize=7.5,
            color="white" if rel > thresh else INK,
        )

### 1.1 Manifest y predicciones

Se leen todos los archivos tabulares bajo `runs/` (`.parquet`, `.jsonl`, `.csv`) y se conservan los que tienen columnas `image_id` y `prediction`.
Si un archivo no trae `exp_id`, se usa el nombre de su carpeta. Las columnas `model` y `prompt` que vienen en los archivos provienen de un
*split* incorrecto del identificador (p. ej. `model = "m3di"`), por lo que se renombran a `*_raw` y se reemplazan por lo parseado en la sección 1.2.

In [ ]:
def canon_id(s: pd.Series) -> pd.Series:
    # Clave canónica de imagen: tolera '0007' vs 7 vs '7'.
    s = s.astype(str).str.strip()
    if s.str.fullmatch(r"\d+").all():
        return s.str.lstrip("0").replace("", "0")
    return s


manifest = pd.read_parquet(MANIFEST)
manifest["image_id"] = manifest["image_id"].astype(str)
manifest["_key"] = canon_id(manifest["image_id"])
assert manifest["_key"].is_unique, "image_id duplicado en el manifest"

ATTR_COLS = [
    "object_shape",
    "object_xpos",
    "object_ypos",
    "object_color",
    "spotlight_color",
    "background_color",
]
missing_attrs = [c for c in ATTR_COLS if c not in manifest.columns]
assert not missing_attrs, f"Faltan atributos en el manifest: {missing_attrs}"

for c in ["object_color", "spotlight_color", "background_color"]:
    lo, hi = manifest[c].min(), manifest[c].max()
    if lo < 0 or hi > 1:
        print(
            f"[aviso] {c} fuera de [0,1] (min={lo:.3f}, max={hi:.3f}); se asume tono y se aplica módulo 1."
        )
        manifest[c] = np.mod(manifest[c], 1.0)

print(f"Manifest: {len(manifest):,} imágenes · columnas: {list(manifest.columns)}")
manifest.head(3)

In [ ]:
def _read_any(path: Path) -> pd.DataFrame:
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    if path.suffix == ".jsonl":
        return pd.read_json(path, lines=True, dtype={"image_id": str})
    if path.suffix == ".csv":
        return pd.read_csv(path, dtype={"image_id": str})
    raise ValueError(path.suffix)


files = sorted(p for ext in ("*.parquet", "*.jsonl", "*.csv") for p in RUNS_DIR.rglob(ext))
frames, skipped = [], []
for f in files:
    try:
        d = _read_any(f)
    except Exception as e:
        skipped.append((str(f.relative_to(ROOT)), f"error de lectura: {e}"))
        continue
    if not {"image_id", "prediction"} <= set(d.columns):
        skipped.append((str(f.relative_to(ROOT)), "sin columnas image_id/prediction"))
        continue
    if "exp_id" not in d.columns:
        d["exp_id"] = f.parent.name
    d["source_file"] = str(f.relative_to(ROOT))
    frames.append(d)

assert frames, f"No se encontraron archivos de predicciones en {RUNS_DIR}"
raw = pd.concat(frames, ignore_index=True)
raw = raw.rename(columns={c: f"{c}_raw" for c in ("model", "prompt") if c in raw.columns})
raw["image_id"] = raw["image_id"].astype(str)
raw["_key"] = canon_id(raw["image_id"])

print(
    f"{len(frames)} archivos de predicciones leídos · {len(raw):,} filas · {raw['exp_id'].nunique()} exp_id distintos"
)
if skipped:
    print(f"{len(skipped)} archivos omitidos:")
    display(pd.DataFrame(skipped, columns=["archivo", "motivo"]))

### 1.2 Parseo de `exp_id`

Formato esperado: `e{k}_{dataset}_{split}_{modelo}_p{j}_{nombre-prompt}[_n{N}][_{sufijo}]`, por ejemplo
`e1_m3di_test_internvl_p0_minimal_n1000`. El modelo se captura de forma no codiciosa hasta el primer `_p{j}_`,
de modo que admite guiones bajos (p. ej. `qwen2_5_vl`); el nombre del prompt también puede contenerlos.
Se verifica además que `prompt_raw` (lo que venía mal parseado) sea consistente con el parseo.

In [ ]:
EXP_RE = re.compile(
    r"^(?P<exp>e\d+)_(?P<dataset>[A-Za-z0-9]+)_(?P<split>train|val|valid|test)_"
    r"(?P<model>.+?)_(?P<prompt_id>p\d+)_(?P<prompt_name>.+?)"
    r"(?:_n(?P<n_samples>\d+)(?:_(?P<suffix>.+))?)?$"
)


def parse_exp_id(exp_id: str) -> dict:
    m = EXP_RE.match(exp_id)
    if m is None:
        return {"exp_id": exp_id, "parse_ok": False}
    d = m.groupdict()
    d.update(exp_id=exp_id, parse_ok=True)
    return d


exp_meta = pd.DataFrame([parse_exp_id(e) for e in sorted(raw["exp_id"].unique())])
bad = exp_meta.loc[~exp_meta["parse_ok"], "exp_id"].tolist()
if bad:
    print(f"[aviso] {len(bad)} exp_id no siguen el formato y se excluyen:", bad)
exp_meta = exp_meta[exp_meta["parse_ok"]].drop(columns="parse_ok")
exp_meta["n_samples"] = pd.to_numeric(exp_meta["n_samples"], errors="coerce").astype("Int64")
exp_meta["prompt"] = exp_meta["prompt_id"] + "_" + exp_meta["prompt_name"]

# Filtro por dataset/split (el manifest corresponde a m3di-test)
if KEEP:
    for k, v in KEEP.items():
        if v is not None:
            dropped = exp_meta.loc[exp_meta[k] != v, "exp_id"].tolist()
            if dropped:
                print(f"Excluidas por {k} != {v!r}: {dropped}")
            exp_meta = exp_meta[exp_meta[k] == v]

display(exp_meta)

In [ ]:
df = raw.merge(exp_meta, on="exp_id", how="inner")

# Consistencia con las columnas mal parseadas que venían en los archivos
if "prompt_raw" in df.columns:
    expected = df["split"] + "_" + df["model"] + "_" + df["prompt"]
    incons = np.array([not str(a).startswith(b) for a, b in zip(df["prompt_raw"], expected)])
    print(f"Filas donde prompt_raw no calza con el parseo: {incons.sum()}")
if "model_raw" in df.columns:
    print("Valores de model_raw (deberían ser el dataset):", df["model_raw"].unique().tolist())

# Etiquetas legibles y orden estable
df["model_label"] = df["model"].map(lambda m: MODEL_LABELS.get(m, m))
df["prompt_label"] = df["prompt"].map(lambda p: PROMPT_LABELS.get(p, p))
multi_exp = df["exp"].nunique() > 1
df["run"] = (
    df["model_label"] + " · " + df["prompt_label"] + ((" · " + df["exp"]) if multi_exp else "")
)

MODEL_ORDER = sorted(df["model_label"].unique())
PROMPT_ORDER = (
    df[["prompt_label", "prompt_id"]]
    .drop_duplicates()
    .assign(_k=lambda t: t["prompt_id"].str[1:].astype(int))
    .sort_values(["_k", "prompt_label"])["prompt_label"]
    .tolist()
)
RUN_ORDER = (
    df[["run", "model_label", "prompt_label", "exp"]]
    .drop_duplicates()
    .assign(
        _m=lambda t: t["model_label"].map(MODEL_ORDER.index),
        _p=lambda t: t["prompt_label"].map(PROMPT_ORDER.index),
    )
    .sort_values(["exp", "_m", "_p"])["run"]
    .tolist()
)
if len(MODEL_ORDER) > len(SERIES):
    print("[aviso] más modelos que colores categóricos; considere agrupar o facetar.")
MODEL_COLORS = dict(zip(MODEL_ORDER, SERIES))
RUN_MODEL = df.drop_duplicates("run").set_index("run")["model_label"].to_dict()
RUN_COLORS = {r: MODEL_COLORS[RUN_MODEL[r]] for r in RUN_ORDER}

print(f"Modelos: {MODEL_ORDER}")
print(f"Prompts: {PROMPT_ORDER}")
print(f"Corridas ({len(RUN_ORDER)}):", *RUN_ORDER, sep="\n  ")

## 2. Controles de integridad

Por corrida se revisa: filas, imágenes únicas, duplicados `(exp_id, image_id)`, imágenes que no existen en el manifest,
imágenes del manifest sin predicción y predicciones vacías o nulas. Los duplicados se resuelven conservando la última fila.
Luego los atributos verdaderos se toman **siempre del manifest**; si los archivos de predicción traían atributos, se verifica que coincidan.

In [ ]:
df["prediction"] = df["prediction"].fillna("").astype(str)
manifest_keys = set(manifest["_key"])

integrity = (
    df.groupby("run", sort=False)
    .agg(
        filas=("image_id", "size"),
        imagenes_unicas=("_key", "nunique"),
        duplicados=("_key", lambda s: int(s.duplicated().sum())),
        fuera_manifest=("_key", lambda s: int((~s.isin(manifest_keys)).sum())),
        vacias=("prediction", lambda s: int(s.str.strip().eq("").sum())),
    )
    .reindex(RUN_ORDER)
)
integrity["sin_prediccion"] = [
    len(manifest_keys - set(df.loc[df["run"] == r, "_key"])) for r in integrity.index
]
integrity["cobertura"] = 1 - integrity["sin_prediccion"] / len(manifest_keys)
display(integrity.style.format({"cobertura": "{:.1%}"}))
save_table(integrity, "integridad_corridas")

n_before = len(df)
df = df.drop_duplicates(["exp_id", "_key"], keep="last")
df = df[df["_key"].isin(manifest_keys)]
print(f"Filas eliminadas por duplicado o fuera del manifest: {n_before - len(df)}")

In [ ]:
# Atributos verdaderos desde el manifest (fuente de verdad)
attr_in_preds = [c for c in ATTR_COLS if c in df.columns]
df = df.merge(manifest[["_key"] + ATTR_COLS], on="_key", how="left", suffixes=("_pred_file", ""))
for c in attr_in_preds:
    a, b = df[f"{c}_pred_file"], df[c]
    diff = ~np.isclose(a.astype(float), b.astype(float), atol=1e-6)
    if diff.any():
        print(
            f"[aviso] {c}: {diff.sum()} filas difieren entre archivo de predicción y manifest (se usa el manifest)"
        )
df = df.drop(columns=[f"{c}_pred_file" for c in attr_in_preds])
df = df.reset_index(drop=True)
print(f"Tabla de análisis: {len(df):,} filas × {df.shape[1]} columnas")

## 3. Distribución de los factores verdaderos

Si la distribución de un factor no es uniforme, la exactitud agregada puede estar dominada por las clases frecuentes;
por eso en la sección 8 se reporta junto a la **línea base mayoritaria** $\max_k \hat\pi_k$ (acertar siempre la clase más frecuente).

In [ ]:
gt = manifest.copy()
fig, axes = plt.subplots(2, 3, figsize=(12, 6.2))

for ax, col, title in zip(
    axes[0],
    ["object_shape", "object_xpos", "object_ypos"],
    ["Forma (código)", "Posición x (código)", "Posición y (código)"],
):
    vc = gt[col].value_counts().sort_index()
    ax.bar(vc.index.astype(str), vc.values, color=SERIES[0], width=0.6, edgecolor=SURF, linewidth=2)
    ax.set_title(title)
    ax.set_ylabel("imágenes")
    ax.grid(axis="x", visible=False)

bins = np.linspace(0, 1, 25)
centers = (bins[:-1] + bins[1:]) / 2
bin_colors = [colorsys.hsv_to_rgb(h, 0.75, 0.9) for h in centers]  # aquí el color ES el dato (tono)
for ax, col, title in zip(
    axes[1],
    ["object_color", "spotlight_color", "background_color"],
    ["Tono del objeto", "Tono del foco", "Tono del fondo"],
):
    counts, _ = np.histogram(gt[col], bins=bins)
    ax.bar(centers, counts, width=bins[1] - bins[0], color=bin_colors, edgecolor=SURF, linewidth=1)
    ax.set_title(title)
    ax.set_xlabel("hue ∈ [0, 1)")
    ax.set_ylabel("imágenes")
    ax.grid(axis="x", visible=False)

fig.suptitle(
    "Factores generativos en M3Di-test", x=0.01, ha="left", fontsize=12, fontweight="semibold"
)
fig.tight_layout()
savefig(fig, "01_factores_verdaderos")
plt.show()

## 4. Léxicos y validación de los extractores

### 4.1 Léxicos

* **Forma.** Siete clases de Multimodal3DIdent con sinónimos semánticamente equivalentes (p. ej. *hare ≈ rabbit ≈ bunny*).
  Se detecta la **primera** clase mencionada y el conjunto de clases mencionadas (para medir alucinación de forma).
* **Color.** El tono verdadero $h \in [0,1)$ se discretiza en 8 categorías circulares por ángulo $\theta = 360\,h$:
  rojo $[345°, 15°)$, naranjo $[15°, 40°)$, amarillo $[40°, 70°)$, verde $[70°, 160°)$, cian $[160°, 195°)$, azul $[195°, 255°)$,
  morado $[255°, 285°)$ y rosado/magenta $[285°, 345°)$. Además de la coincidencia exacta se reporta la **tolerancia de una categoría adyacente**,
  porque los bordes son arbitrarios (p. ej. $\theta = 82°$ puede describirse legítimamente como amarillo o verde).
* **Posición.** Horizontal $\{\text{left}, \text{center}, \text{right}\}$ y vertical $\{\text{top}, \text{center}, \text{bottom}\}$; *center/middle*
  completa el eje que no fue mencionado explícitamente.

### 4.2 Asignación de colores a entidades

Una descripción suele nombrar varios colores (objeto, fondo, luz). Cada mención cromática se asigna según su contexto dentro de la oración:
si en las 3 palabras siguientes (o, en su defecto, en las 3 anteriores) aparece un sustantivo de fondo (*background, backdrop, wall, floor, …*)
se asigna al **fondo**; si aparece uno de iluminación (*light, spotlight, glow, …*) al **foco**; en otro caso al **objeto**.
Los colores coordinados (*"blue and purple"*, *"blue-green"*) heredan la asignación del anterior. El color predicho de cada entidad es la primera mención asignada a ella.

In [ ]:
SHAPE_LEXICON = {
    "teapot": ["teapot", "tea pot", "kettle"],
    "hare": ["hare", "rabbit", "bunny", "bunnies"],
    "dragon": ["dragon"],
    "cow": ["cow", "bull", "ox", "oxen", "cattle", "calf"],
    "armadillo": ["armadillo"],
    "horse": ["horse", "pony", "stallion", "mare"],
    "head": ["head", "bust", "face"],
}
SHAPE_CANON_ORDER = list(
    SHAPE_LEXICON
)  # orden de Multimodal3DIdent (se valida empíricamente abajo)


def _term_regex(term):
    # Palabra completa con plural opcional; permite espacio o guion en términos compuestos.
    t = re.escape(term).replace(r"\ ", r"[\s\-]?")
    return rf"\b{t}(?:s|es)?\b"


SHAPE_RE = {
    k: re.compile("|".join(_term_regex(t) for t in v), re.IGNORECASE)
    for k, v in SHAPE_LEXICON.items()
}


def find_shapes(text):
    # Lista ordenada por posición de (inicio, clase) para todas las menciones de forma.
    hits = [(m.start(), k) for k, rx in SHAPE_RE.items() for m in rx.finditer(text)]
    return sorted(hits)


COLOR_CATS = [
    "red",
    "orange",
    "yellow",
    "green",
    "cyan",
    "blue",
    "purple",
    "pink",
]  # orden circular
HUE_EDGES_DEG = [15, 40, 70, 160, 195, 255, 285, 345]  # límite superior de cada categoría
COLOR_LEXICON = {
    "red": ["red", "reddish", "crimson", "scarlet", "maroon", "burgundy", "ruby", "cherry"],
    "orange": [
        "orange",
        "orangish",
        "amber",
        "tangerine",
        "peach",
        "rust",
        "copper",
        "salmon",
        "coral",
    ],
    "yellow": ["yellow", "yellowish", "gold", "golden", "mustard", "lemon"],
    "green": ["green", "greenish", "lime", "olive", "chartreuse", "emerald", "mint"],
    "cyan": ["cyan", "teal", "turquoise", "aqua", "aquamarine"],
    "blue": ["blue", "bluish", "navy", "azure", "cobalt", "sapphire", "cerulean"],
    "purple": ["purple", "purplish", "violet", "lavender", "indigo", "lilac", "plum", "mauve"],
    "pink": ["pink", "pinkish", "magenta", "fuchsia", "rose", "hot pink"],
    "achromatic": ["white", "black", "gray", "grey", "silver", "brown", "beige", "tan"],
}
COLOR_TERM_RE = re.compile(
    r"\b("
    + "|".join(
        sorted((re.escape(t) for v in COLOR_LEXICON.values() for t in v), key=len, reverse=True)
    )
    + r")\b",
    re.IGNORECASE,
)
TERM2CAT = {t: c for c, v in COLOR_LEXICON.items() for t in v}

BG_WORDS = {
    "background",
    "backdrop",
    "backgrounds",
    "wall",
    "walls",
    "floor",
    "ground",
    "surface",
    "sky",
    "surroundings",
    "environment",
    "gradient",
    "setting",
    "canvas",
    "plane",
}
LIGHT_WORDS = {
    "light",
    "lights",
    "lighting",
    "lit",
    "spotlight",
    "spotlights",
    "glow",
    "glowing",
    "illumination",
    "illuminated",
    "illuminating",
    "beam",
    "beams",
    "shadow",
    "shadows",
    "highlight",
    "highlights",
    "reflection",
    "reflections",
    "hue",
    "tint",
    "shine",
    "shining",
}
WORD_RE = re.compile(r"[a-z]+(?:'[a-z]+)?")


def hue_to_cat(h):
    deg = (float(h) * 360.0) % 360.0
    if deg >= 345 or deg < 15:
        return "red"
    for cat, upper in zip(COLOR_CATS[1:], HUE_EDGES_DEG[1:]):
        if deg < upper:
            return cat
    return "red"


def cat_distance(a, b):
    # Distancia circular entre categorías cromáticas (inf si alguna es acromática o nula).
    if a not in COLOR_CATS or b not in COLOR_CATS:
        return np.inf
    d = abs(COLOR_CATS.index(a) - COLOR_CATS.index(b))
    return min(d, len(COLOR_CATS) - d)


def circ_diff(a, b):
    # Diferencia circular con signo en [-0.5, 0.5) entre dos tonos en [0,1).
    return (np.asarray(a) - np.asarray(b) + 0.5) % 1.0 - 0.5


def assign_colors(text):
    # Devuelve la lista de (categoría, entidad) en orden de aparición; entidad ∈ {object, background, spotlight}.
    out = []
    for sent in re.split(r"(?<=[.!?;])\s+|\n+", text.lower()):
        prev_end, prev_ent = None, None
        for m in COLOR_TERM_RE.finditer(sent):
            cat = TERM2CAT[m.group(1).lower()]
            gap = sent[prev_end : m.start()] if prev_end is not None else None
            if gap is not None and re.fullmatch(
                r"\s*(?:,|and|or|to|-|/|and\s+light|and\s+dark)?\s*", gap
            ):
                ent = prev_ent  # color coordinado: hereda
            else:
                after = WORD_RE.findall(sent[m.end() :])[:3]
                before = WORD_RE.findall(sent[: m.start()])[-3:]
                if any(w in BG_WORDS for w in after):
                    ent = "background"
                elif any(w in LIGHT_WORDS for w in after):
                    ent = "spotlight"
                elif any(w in BG_WORDS for w in before):
                    ent = "background"
                elif any(w in LIGHT_WORDS for w in before):
                    ent = "spotlight"
                else:
                    ent = "object"
            out.append((cat, ent))
            prev_end, prev_ent = m.end(), ent
    return out


POS_STRIP = re.compile(
    r"\bon top of\b|\bright[\s\-]angled?\b|\ball right\b|\bright now\b|\bleft over\b|\bleftover\b",
    re.IGNORECASE,
)


def parse_position(text):
    t = POS_STRIP.sub(" ", text.lower())
    has_left, has_right = bool(re.search(r"\bleft\b", t)), bool(re.search(r"\bright\b", t))
    has_top = bool(re.search(r"\b(top|upper)\b", t))
    has_bottom = bool(re.search(r"\b(bottom|lower)\b", t))
    has_center = bool(re.search(r"\b(center|centre|centered|centred|middle|central)\b", t))
    h = (
        "ambiguous"
        if (has_left and has_right)
        else "left"
        if has_left
        else "right"
        if has_right
        else None
    )
    v = (
        "ambiguous"
        if (has_top and has_bottom)
        else "top"
        if has_top
        else "bottom"
        if has_bottom
        else None
    )
    if has_center:
        h = h or "center"
        v = v or "center"
    return h, v

### 4.3 Calibración de códigos con el texto de referencia

Los códigos enteros de forma y posición no traen nombre. En lugar de suponer un orden, se infiere empíricamente la biyección
código → etiqueta a partir del texto de referencia (que describe los factores explícitamente) y se reporta la **pureza**
$\max_k \hat P(\text{etiqueta}=k \mid \text{código})$ como medida de precisión del extractor. Una pureza cercana a 1 respalda
usar el mismo extractor sobre las predicciones.

In [ ]:
refs = df.drop_duplicates("_key")[["_key", "reference"] + ATTR_COLS].reset_index(drop=True)
refs["reference"] = refs["reference"].fillna("").astype(str)
refs["ref_shape"] = refs["reference"].map(lambda t: (find_shapes(t) or [(None, None)])[0][1])
refs[["ref_h", "ref_v"]] = refs["reference"].map(parse_position).apply(pd.Series)


def code_map(codes, labels, name):
    ct = pd.crosstab(codes, labels.fillna("∅"))
    purity = (ct.max(axis=1) / ct.sum(axis=1)).rename("pureza")
    mapping = ct.drop(columns="∅", errors="ignore").idxmax(axis=1)
    print(f"{name}: pureza media ponderada = {(ct.max(axis=1).sum() / ct.values.sum()):.3f}")
    display(pd.concat([ct, purity], axis=1).style.format({"pureza": "{:.3f}"}).set_caption(name))
    if mapping.duplicated().any():
        print(f"[aviso] {name}: la asignación código→etiqueta no es biyectiva: {mapping.to_dict()}")
    return mapping.to_dict()


SHAPE_NAMES = code_map(
    refs["object_shape"], refs["ref_shape"], "Forma: código vs. término en la referencia"
)
XPOS_NAMES = code_map(
    refs["object_xpos"], refs["ref_h"], "Posición x: código vs. término en la referencia"
)
YPOS_NAMES = code_map(
    refs["object_ypos"], refs["ref_v"], "Posición y: código vs. término en la referencia"
)

# Respaldo: si algún código no quedó mapeado se usa el orden canónico de M3Di
for c in sorted(manifest["object_shape"].unique()):
    SHAPE_NAMES.setdefault(c, SHAPE_CANON_ORDER[c] if c < len(SHAPE_CANON_ORDER) else str(c))
print("SHAPE_NAMES =", SHAPE_NAMES)
print("XPOS_NAMES  =", XPOS_NAMES)
print("YPOS_NAMES  =", YPOS_NAMES)

### 4.4 ¿Es `object_color` un tono en $[0,1)$?

Las referencias nombran el color con nombres de Matplotlib (`xkcd:…`, `tab:…`). Convirtiendo esos nombres a HSV se obtiene un tono
$h^{\text{ref}}$ que puede compararse con el factor $h^{\text{obj}}$ mediante la diferencia circular
$\Delta = \big((h^{\text{ref}} - h^{\text{obj}} + \tfrac12) \bmod 1\big) - \tfrac12$. Si $|\Delta|$ es pequeño, se valida que el factor es un tono
y que la discretización de la sección 4.1 es coherente con lo que el propio generador de texto nombra.

In [ ]:
NAMED_COLOR_QUOTED = re.compile(r'["“]((?:xkcd|tab|css):[^"”]+)["”]', re.IGNORECASE)
NAMED_COLOR_BARE = re.compile(r"\b((?:xkcd|tab|css):[a-z0-9\-]+)", re.IGNORECASE)


def ref_named_hue(text):
    m = NAMED_COLOR_QUOTED.search(text) or NAMED_COLOR_BARE.search(text)
    if not m:
        return np.nan, None
    name = m.group(1).strip().lower().replace("css:", "")
    try:
        r, g, b = mcolors.to_rgb(name)
    except ValueError:
        return np.nan, name
    return colorsys.rgb_to_hsv(r, g, b)[0], name


refs[["ref_hue", "ref_color_name"]] = refs["reference"].map(ref_named_hue).apply(pd.Series)
ok = refs["ref_hue"].notna()
delta_deg = 360 * circ_diff(refs.loc[ok, "ref_hue"].astype(float), refs.loc[ok, "object_color"])
print(f"Referencias con color nombrado reconocible: {ok.mean():.1%}")
print(
    f"|Δ| mediana = {np.median(np.abs(delta_deg)):.1f}°, P90 = {np.percentile(np.abs(delta_deg), 90):.1f}°"
)
agree = refs.loc[ok, "ref_hue"].map(hue_to_cat) == refs.loc[ok, "object_color"].map(hue_to_cat)
agree_adj = [
    cat_distance(a, b) <= 1
    for a, b in zip(
        refs.loc[ok, "ref_hue"].map(hue_to_cat), refs.loc[ok, "object_color"].map(hue_to_cat)
    )
]
print(
    f"Categoría(ref) = categoría(h_obj): exacta {agree.mean():.1%} · ±1 categoría {np.mean(agree_adj):.1%}"
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ax = axes[0]
ax.scatter(
    refs.loc[ok, "object_color"],
    refs.loc[ok, "ref_hue"],
    s=10,
    alpha=0.5,
    color=SERIES[0],
    linewidths=0,
)
ax.plot([0, 1], [0, 1], color=INK3, lw=1, ls="--")
ax.set(
    xlabel="$h^{obj}$ (factor)",
    ylabel="$h^{ref}$ (nombre en la referencia)",
    title="Tono del factor vs. tono del nombre",
    xlim=(0, 1),
    ylim=(0, 1),
)
ax = axes[1]
ax.hist(delta_deg, bins=np.arange(-180, 181, 7.5), color=SERIES[0], edgecolor=SURF, linewidth=1)
ax.axvline(0, color=INK3, lw=1, ls="--")
ax.set(xlabel="Δ circular (grados)", ylabel="imágenes", title="Distribución de Δ")
ax.grid(axis="x", visible=False)
fig.tight_layout()
savefig(fig, "02_validacion_tono_referencia")
plt.show()

## 5. Extracción de atributos desde las predicciones

Para cada fila y atributo $a$ se construyen: `a_pred` (valor extraído o nulo), `a_true`, `a_mentioned` $=\mathbb 1[\hat z_a \neq \varnothing]$
y `a_correct` $=\mathbb 1[\hat z_a = z_a]$ (una omisión cuenta como error). Para colores se agrega `a_adj` (acierto con tolerancia de una categoría).

In [ ]:
STOP = set(
    [
        "a",
        "an",
        "the",
        "of",
        "and",
        "or",
        "in",
        "on",
        "at",
        "to",
        "is",
        "are",
        "was",
        "were",
        "be",
        "been",
        "being",
        "it",
        "its",
        "this",
        "that",
        "with",
        "as",
        "by",
        "for",
        "from",
        "into",
        "onto",
        "over",
        "under",
        "which",
        "while",
        "there",
        "their",
        "they",
        "them",
        "image",
        "picture",
        "shows",
        "show",
        "showing",
        "depicts",
        "depicting",
        "features",
        "featuring",
        "appears",
        "appear",
        "seems",
        "seem",
        "has",
        "have",
        "having",
        "can",
        "could",
        "would",
        "will",
        "also",
        "very",
        "some",
        "any",
        "one",
        "two",
        "three",
        "more",
        "most",
        "than",
        "then",
        "render",
        "rendered",
        "rendering",
        "3d",
        "scene",
        "view",
        "photo",
        "object",
    ]
)
COLOR_WORDS = set(TERM2CAT)


def text_stats(t):
    words = WORD_RE.findall(t.lower())
    n = len(words)
    tri = list(zip(words, words[1:], words[2:]))
    rep = 1 - len(set(tri)) / len(tri) if tri else 0.0
    sents = [s for s in re.split(r"(?<=[.!?])\s+", t.strip()) if s]
    return n, len(t), len(sents), rep, bool(re.search(r'[.!?)"\']\s*$', t.strip()))


def content_tokens(t):
    t = re.sub(r"\b(?:xkcd|tab|css):", " ", t.lower())
    return [w for w in WORD_RE.findall(t) if w not in STOP]


def unigram_f1(pred, ref):
    p, r = Counter(content_tokens(pred)), Counter(content_tokens(ref))
    ov = sum((p & r).values())
    if ov == 0:
        return 0.0
    prec, rec = ov / sum(p.values()), ov / sum(r.values())
    return 2 * prec * rec / (prec + rec)


def extract_row(pred):
    shapes = find_shapes(pred)
    h, v = parse_position(pred)
    cols = assign_colors(pred)
    first = lambda ent: next((c for c, e in cols if e == ent), None)
    n_w, n_c, n_s, rep, ends = text_stats(pred)
    return {
        "shape_pred": shapes[0][1] if shapes else None,
        "shape_set": sorted({k for _, k in shapes}),
        "xpos_pred": h,
        "ypos_pred": v,
        "color_obj_pred": first("object"),
        "color_bg_pred": first("background"),
        "color_spot_pred": first("spotlight"),
        "n_color_mentions": len(cols),
        "n_words": n_w,
        "n_chars": n_c,
        "n_sents": n_s,
        "rep_trigram": rep,
        "ends_clean": ends,
    }


ext = pd.DataFrame([extract_row(p) for p in df["prediction"]], index=df.index)
df = pd.concat([df.drop(columns=[c for c in ext.columns if c in df.columns]), ext], axis=1)

# Valores verdaderos en el mismo espacio de etiquetas
df["shape_true"] = df["object_shape"].map(SHAPE_NAMES)
df["xpos_true"] = df["object_xpos"].map(XPOS_NAMES)
df["ypos_true"] = df["object_ypos"].map(YPOS_NAMES)
df["color_obj_true"] = df["object_color"].map(hue_to_cat)
df["color_bg_true"] = df["background_color"].map(hue_to_cat)
df["color_spot_true"] = df["spotlight_color"].map(hue_to_cat)

ATTRS = ["shape", "xpos", "ypos", "color_obj", "color_bg", "color_spot"]
ATTR_LABELS = {
    "shape": "Forma",
    "xpos": "Posición x",
    "ypos": "Posición y",
    "pos_joint": "Posición (x,y)",
    "color_obj": "Color objeto",
    "color_obj_adj": "Color objeto ±1",
    "color_bg": "Color fondo",
    "color_bg_adj": "Color fondo ±1",
    "color_spot": "Color foco",
    "color_spot_adj": "Color foco ±1",
    "core": "Forma+pos+color",
}
for a in ATTRS:
    df[f"{a}_mentioned"] = df[f"{a}_pred"].notna()
    df[f"{a}_correct"] = df[f"{a}_pred"].eq(df[f"{a}_true"]) & df[f"{a}_mentioned"]
for a in ["color_obj", "color_bg", "color_spot"]:
    df[f"{a}_adj_mentioned"] = df[f"{a}_mentioned"]
    df[f"{a}_adj_correct"] = [
        cat_distance(p, t) <= 1 for p, t in zip(df[f"{a}_pred"], df[f"{a}_true"])
    ]
df["pos_joint_mentioned"] = df["xpos_mentioned"] & df["ypos_mentioned"]
df["pos_joint_correct"] = df["xpos_correct"] & df["ypos_correct"]
df["core_mentioned"] = df["shape_mentioned"] & df["pos_joint_mentioned"] & df["color_obj_mentioned"]
df["core_correct"] = df["shape_correct"] & df["pos_joint_correct"] & df["color_obj_correct"]

# Alucinación de forma: menciona al menos una clase distinta de la verdadera
df["shape_halluc"] = [any(s != t for s in S) for S, t in zip(df["shape_set"], df["shape_true"])]
df["ref_n_words"] = df["reference"].fillna("").map(lambda t: len(WORD_RE.findall(t.lower())))
df["unigram_f1"] = [unigram_f1(p, r) for p, r in zip(df["prediction"], df["reference"].fillna(""))]

df[
    [
        "run",
        "image_id",
        "prediction",
        "shape_pred",
        "shape_true",
        "xpos_pred",
        "xpos_true",
        "ypos_pred",
        "ypos_true",
        "color_obj_pred",
        "color_obj_true",
        "color_bg_pred",
        "color_bg_true",
    ]
].head(8)

## 6. Longitud, truncamiento y degeneración del texto

* **Truncamiento probable:** el texto no termina en puntuación de cierre, síntoma típico de alcanzar `max_new_tokens`.
* **Repetición:** $\rho = 1 - \dfrac{|\{\text{trigramas distintos}\}|}{|\{\text{trigramas}\}|}$; valores altos indican bucles degenerados.
* **F1 unigrama vs. referencia:** solapamiento léxico de palabras de contenido; es sólo un descriptor (no mide fidelidad visual).

In [ ]:
def q(p):
    f = lambda s: s.quantile(p)
    f.__name__ = f"p{int(p * 100)}"
    return f


length_tab = (
    df.groupby("run", sort=False)
    .agg(
        n=("n_words", "size"),
        palabras_media=("n_words", "mean"),
        palabras_de=("n_words", "std"),
        palabras_p50=("n_words", q(0.5)),
        palabras_p95=("n_words", q(0.95)),
        oraciones_media=("n_sents", "mean"),
        vacias=("n_words", lambda s: (s == 0).mean()),
        truncadas=("ends_clean", lambda s: 1 - s.mean()),
        repeticion_media=("rep_trigram", "mean"),
        f1_unigrama=("unigram_f1", "mean"),
    )
    .reindex(RUN_ORDER)
)
display(
    length_tab.style.format(
        {c: "{:.1%}" for c in ["vacias", "truncadas"]}
        | {
            c: "{:.2f}"
            for c in [
                "palabras_media",
                "palabras_de",
                "oraciones_media",
                "repeticion_media",
                "f1_unigrama",
            ]
        }
    )
)
save_table(length_tab, "longitud_por_corrida")
print(
    f"Referencia: {refs['reference'].map(lambda t: len(WORD_RE.findall(t.lower()))).median():.0f} palabras (mediana)"
)

In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 0.45 * len(RUN_ORDER) + 1.6),
    sharey=True,
    gridspec_kw={"width_ratios": [2.2, 1]},
)
ax = axes[0]
data = [df.loc[df["run"] == r, "n_words"].values for r in RUN_ORDER]
bp = ax.boxplot(
    data,
    vert=False,
    widths=0.55,
    patch_artist=True,
    showfliers=True,
    medianprops=dict(color=INK, lw=1.5),
    whiskerprops=dict(color=INK3),
    capprops=dict(color=INK3),
    flierprops=dict(marker="o", ms=2.5, mfc=INK3, mec="none", alpha=0.5),
)
for patch, r in zip(bp["boxes"], RUN_ORDER):
    patch.set(facecolor=RUN_COLORS[r], alpha=0.75, edgecolor=RUN_COLORS[r])
ref_med = refs["reference"].map(lambda t: len(WORD_RE.findall(t.lower()))).median()
ax.axvline(ref_med, color=INK2, ls="--", lw=1)
ax.text(ref_med, len(RUN_ORDER) + 0.42, " mediana referencia", color=INK2, fontsize=8, va="bottom")
ax.set_yticks(range(1, len(RUN_ORDER) + 1), RUN_ORDER)
ax.invert_yaxis()
ax.set(xlabel="palabras por descripción", title="Longitud de las predicciones")
ax.grid(axis="y", visible=False)

ax = axes[1]
y = np.arange(1, len(RUN_ORDER) + 1)
ax.barh(
    y - 0.18,
    length_tab["truncadas"],
    height=0.34,
    color=[RUN_COLORS[r] for r in RUN_ORDER],
    label="truncadas",
)
ax.barh(
    y + 0.18,
    length_tab["vacias"],
    height=0.34,
    color=[RUN_COLORS[r] for r in RUN_ORDER],
    alpha=0.35,
    label="vacías",
)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set(title="Truncadas / vacías", xlabel="proporción")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=2)
ax.grid(axis="y", visible=False)
fig.tight_layout()
savefig(fig, "03_longitud_truncamiento")
plt.show()

## 7. Vocabulario

* **Diversidad léxica** por corrida: distinct-$n$ $= \dfrac{|\{n\text{-gramas distintos}\}|}{|\{n\text{-gramas}\}|}$ sobre todo el corpus de la corrida (bajo = descripciones plantilla).
* **Términos más frecuentes** por modelo (sin *stopwords*).
* **¿Cómo llama el modelo al objeto cuando no usa ninguna palabra del léxico de forma?** Se listan los términos más frecuentes en esas predicciones, por forma verdadera; esto revela confusiones sistemáticas (p. ej. *dragon → dinosaur*) y sinónimos que convendría agregar al léxico.

In [ ]:
def distinct_n(texts, n):
    grams = [g for t in texts for g in zip(*[WORD_RE.findall(t.lower())[i:] for i in range(n)])]
    return len(set(grams)) / len(grams) if grams else np.nan


vocab_tab = pd.DataFrame(
    {
        "vocabulario": df.groupby("run", sort=False)["prediction"].apply(
            lambda s: len({w for t in s for w in WORD_RE.findall(t.lower())})
        ),
        "distinct_1": df.groupby("run", sort=False)["prediction"].apply(lambda s: distinct_n(s, 1)),
        "distinct_2": df.groupby("run", sort=False)["prediction"].apply(lambda s: distinct_n(s, 2)),
        "predicciones_unicas": df.groupby("run", sort=False)["prediction"].apply(
            lambda s: s.nunique() / len(s)
        ),
        "menciones_color_media": df.groupby("run", sort=False)["n_color_mentions"].mean(),
    }
).reindex(RUN_ORDER)
display(
    vocab_tab.style.format(
        "{:.3f}",
        subset=["distinct_1", "distinct_2", "predicciones_unicas", "menciones_color_media"],
    )
)
save_table(vocab_tab, "vocabulario_por_corrida")

top_terms = {
    m: Counter(
        w for t in df.loc[df["model_label"] == m, "prediction"] for w in content_tokens(t)
    ).most_common(15)
    for m in MODEL_ORDER
}
top_df = pd.DataFrame(
    {m: [f"{w} ({c})" for w, c in v] + [""] * (15 - len(v)) for m, v in top_terms.items()}
)
top_df.index = range(1, 16)
display(top_df)

In [ ]:
LEX_WORDS = (
    {w for v in SHAPE_LEXICON.values() for w in v}
    | COLOR_WORDS
    | BG_WORDS
    | LIGHT_WORDS
    | {
        "left",
        "right",
        "top",
        "bottom",
        "center",
        "centre",
        "middle",
        "upper",
        "lower",
        "positioned",
        "located",
        "placed",
        "side",
        "corner",
        "dark",
        "light",
        "bright",
        "small",
        "large",
        "figure",
        "model",
        "sculpture",
    }
)
undetected = df[~df["shape_mentioned"]]
print(
    f"Predicciones sin ninguna forma del léxico: {len(undetected):,} ({len(undetected) / len(df):.1%})"
)
rows = {}
for s in [SHAPE_NAMES[k] for k in sorted(SHAPE_NAMES)]:
    c = Counter(
        w
        for t in undetected.loc[undetected["shape_true"] == s, "prediction"]
        for w in content_tokens(t)
        if w not in LEX_WORDS
    )
    rows[s] = [f"{w} ({n})" for w, n in c.most_common(10)] + [""] * (10 - len(c.most_common(10)))
alias_df = pd.DataFrame(rows, index=range(1, 11))
display(alias_df)
save_table(alias_df, "terminos_forma_no_detectada", index=False)

## 8. Fidelidad por atributo

Para una corrida $r$ y un atributo $a$ con $N_r$ imágenes se reportan tres cantidades:

$$
\underbrace{\hat c_{r,a} = \frac{1}{N_r}\sum_{i}\mathbb 1[\hat z_{i,a} \neq \varnothing]}_{\text{tasa de mención}},\qquad
\underbrace{\hat\alpha_{r,a} = \frac{1}{N_r}\sum_{i}\mathbb 1[\hat z_{i,a} = z_{i,a}]}_{\text{exactitud (omisión = error)}},\qquad
\underbrace{\hat\alpha^{\mid}_{r,a} = \frac{\hat\alpha_{r,a}}{\hat c_{r,a}}}_{\text{exactitud condicional a mención}}.
$$

$\hat\alpha$ penaliza tanto la omisión como el error; $\hat\alpha^{\mid}$ aísla la calidad de lo que sí se dice.
Los intervalos al $1-\alpha$ usan la aproximación de **Wilson**, preferible a Wald con proporciones cercanas a 0 o 1:

$$
\tilde p \pm \frac{z_{1-\alpha/2}}{1+z^2/n}\sqrt{\frac{\hat p(1-\hat p)}{n} + \frac{z^2}{4n^2}},\qquad
\tilde p = \frac{\hat p + z^2/(2n)}{1+z^2/n}.
$$

In [ ]:
def wilson(k, n, alpha=ALPHA):
    if n == 0:
        return np.nan, np.nan
    z = stats.norm.ppf(1 - alpha / 2)
    p = k / n
    den = 1 + z**2 / n
    c = (p + z**2 / (2 * n)) / den
    h = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / den
    return c - h, c + h


METRIC_ATTRS = [
    "shape",
    "xpos",
    "ypos",
    "pos_joint",
    "color_obj",
    "color_obj_adj",
    "color_bg",
    "color_bg_adj",
    "color_spot",
    "color_spot_adj",
    "core",
]
rows = []
for r in RUN_ORDER:
    g = df[df["run"] == r]
    n = len(g)
    for a in METRIC_ATTRS:
        k, m = int(g[f"{a}_correct"].sum()), int(g[f"{a}_mentioned"].sum())
        lo, hi = wilson(k, n)
        rows.append(
            dict(
                run=r,
                model=RUN_MODEL[r],
                prompt=g["prompt_label"].iat[0],
                attr=a,
                n=n,
                mencion=m / n,
                exactitud=k / n,
                ic_lo=lo,
                ic_hi=hi,
                exactitud_cond=k / m if m else np.nan,
            )
        )
acc = pd.DataFrame(rows)

# Línea base mayoritaria sobre la distribución verdadera
base_src = df.drop_duplicates("_key")
majority = {a: base_src[f"{a}_true"].value_counts(normalize=True).iloc[0] for a in ATTRS}
majority["pos_joint"] = (
    (base_src["xpos_true"] + "|" + base_src["ypos_true"]).value_counts(normalize=True).iloc[0]
)
for a in ["color_obj", "color_bg", "color_spot"]:
    vc = base_src[f"{a}_true"].value_counts(normalize=True)
    majority[f"{a}_adj"] = max(
        sum(vc.get(c2, 0) for c2 in COLOR_CATS if cat_distance(c1, c2) <= 1) for c1 in COLOR_CATS
    )
majority["core"] = np.nan

acc_wide = acc.pivot(index="run", columns="attr", values="exactitud").reindex(
    index=RUN_ORDER, columns=METRIC_ATTRS
)
acc_wide.columns = [ATTR_LABELS[c] for c in acc_wide.columns]
display(
    acc_wide.style.format("{:.3f}")
    .background_gradient(cmap=SEQ, vmin=0, vmax=1)
    .set_caption("Exactitud (omisión = error) por corrida y atributo")
)
save_table(acc, "fidelidad_por_atributo_largo", index=False)
save_table(acc_wide, "fidelidad_por_atributo")
print(
    "Línea base mayoritaria:",
    {ATTR_LABELS[k]: round(float(v), 3) for k, v in majority.items() if not np.isnan(v)},
)

In [ ]:
# Mención vs. exactitud condicional: separa "no lo dice" de "lo dice mal"
fig, axes = plt.subplots(1, 3, figsize=(15, 0.42 * len(RUN_ORDER) + 1.8), sharey=True)
for ax, metric, title in zip(
    axes,
    ["mencion", "exactitud", "exactitud_cond"],
    [
        "Tasa de mención $\\hat c$",
        "Exactitud $\\hat\\alpha$",
        "Exactitud condicional $\\hat\\alpha^{\\mid}$",
    ],
):
    sub = ["shape", "xpos", "ypos", "color_obj", "color_bg", "color_spot"]
    M = (
        acc.pivot(index="run", columns="attr", values=metric)
        .reindex(index=RUN_ORDER, columns=sub)
        .values
    )
    ax.pcolormesh(M, cmap=SEQ, vmin=0, vmax=1, edgecolors=SURF, linewidth=2)
    annotate_heatmap(ax, M)
    ax.set_xticks(np.arange(len(sub)) + 0.5, [ATTR_LABELS[a] for a in sub], rotation=30, ha="right")
    ax.set_yticks(np.arange(len(RUN_ORDER)) + 0.5, RUN_ORDER)
    ax.invert_yaxis()
    ax.grid(False)
    ax.set_title(title)
    for s in ax.spines.values():
        s.set_visible(False)
fig.tight_layout()
savefig(fig, "04_heatmap_mencion_exactitud")
plt.show()

In [ ]:
# Exactitud con IC de Wilson por atributo (pequeños múltiplos) + línea base mayoritaria
plot_attrs = ["shape", "pos_joint", "color_obj", "color_obj_adj", "color_bg_adj", "core"]
fig, axes = plt.subplots(
    1,
    len(plot_attrs),
    figsize=(2.6 * len(plot_attrs) + 2.5, 0.4 * len(RUN_ORDER) + 1.6),
    sharey=True,
)
y = np.arange(len(RUN_ORDER))
for ax, a in zip(axes, plot_attrs):
    s = acc[acc["attr"] == a].set_index("run").reindex(RUN_ORDER)
    for yi, r in zip(y, RUN_ORDER):
        ax.plot(
            [s.at[r, "ic_lo"], s.at[r, "ic_hi"]],
            [yi, yi],
            color=RUN_COLORS[r],
            lw=2,
            solid_capstyle="round",
        )
        ax.plot(s.at[r, "exactitud"], yi, "o", ms=7, color=RUN_COLORS[r], mec=SURF, mew=1.5)
    if not np.isnan(majority.get(a, np.nan)):
        ax.axvline(majority[a], color=INK3, ls=":", lw=1.2)
    ax.set_xlim(0, 1)
    ax.set_title(ATTR_LABELS[a], fontsize=10)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
    ax.grid(axis="y", visible=False)
axes[0].set_yticks(y, RUN_ORDER)
axes[0].invert_yaxis()
handles = [
    plt.Line2D([], [], marker="o", ls="", color=MODEL_COLORS[m], label=m) for m in MODEL_ORDER
]
handles.append(plt.Line2D([], [], color=INK3, ls=":", label="línea base mayoritaria"))
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.legend(handles=handles, loc="upper center", ncol=len(handles), bbox_to_anchor=(0.5, 0.95))
fig.suptitle(
    f"Exactitud por atributo (IC Wilson {1 - ALPHA:.0%})", y=1.0, fontsize=12, fontweight="semibold"
)
savefig(fig, "05_exactitud_ic_wilson")
plt.show()

In [ ]:
# Alucinación de forma: menciona clases del léxico distintas de la verdadera
hall = (
    df.groupby("run", sort=False)
    .agg(
        alucinacion_forma=("shape_halluc", "mean"),
        n_formas_mencionadas=("shape_set", lambda s: s.map(len).mean()),
        forma_correcta=("shape_correct", "mean"),
    )
    .reindex(RUN_ORDER)
)
display(hall.style.format("{:.3f}"))
save_table(hall, "alucinacion_forma")

## 9. Análisis de errores

### 9.1 Forma: matrices de confusión

Filas normalizadas: $\hat P(\hat s = k \mid s = j)$. La columna **∅** corresponde a predicciones sin ninguna forma del léxico.

In [ ]:
SHAPE_ORDER = [SHAPE_NAMES[k] for k in sorted(SHAPE_NAMES)]
COLS = SHAPE_ORDER + ["∅"]
ncol = min(3, len(RUN_ORDER))
nrow = int(np.ceil(len(RUN_ORDER) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4.4 * ncol, 3.9 * nrow), squeeze=False)
for ax, r in zip(axes.flat, RUN_ORDER):
    g = df[df["run"] == r]
    ct = pd.crosstab(g["shape_true"], g["shape_pred"].fillna("∅")).reindex(
        index=SHAPE_ORDER, columns=COLS, fill_value=0
    )
    M = (ct.T / ct.sum(axis=1).replace(0, np.nan)).T.values
    ax.pcolormesh(M, cmap=SEQ, vmin=0, vmax=1, edgecolors=SURF, linewidth=1.5)
    annotate_heatmap(ax, M, fmt="{:.2f}")
    ax.set_xticks(np.arange(len(COLS)) + 0.5, COLS, rotation=40, ha="right", fontsize=8)
    ax.set_yticks(np.arange(len(SHAPE_ORDER)) + 0.5, SHAPE_ORDER, fontsize=8)
    ax.invert_yaxis()
    ax.grid(False)
    ax.set_title(r, fontsize=9.5)
    for s in ax.spines.values():
        s.set_visible(False)
for ax in axes.flat[len(RUN_ORDER) :]:
    ax.axis("off")
fig.supxlabel("forma predicha")
fig.supylabel("forma verdadera")
fig.tight_layout()
savefig(fig, "06_confusion_forma")
plt.show()

### 9.2 Posición: exactitud conjunta en la grilla $3\times3$

Cada celda es la exactitud de $(\hat x, \hat y)$ para las imágenes cuyo objeto está en esa posición de la grilla. Permite detectar sesgos
espaciales (p. ej. el modelo describe todo como *"in the center"*).

In [ ]:
X_ORDER = [XPOS_NAMES[k] for k in sorted(XPOS_NAMES)]
Y_ORDER = [YPOS_NAMES[k] for k in sorted(YPOS_NAMES)]
fig, axes = plt.subplots(nrow, ncol, figsize=(3.6 * ncol, 3.3 * nrow), squeeze=False)
for ax, r in zip(axes.flat, RUN_ORDER):
    g = df[df["run"] == r]
    M = (
        g.pivot_table(
            index="ypos_true", columns="xpos_true", values="pos_joint_correct", aggfunc="mean"
        )
        .reindex(index=Y_ORDER, columns=X_ORDER)
        .values
    )
    ax.pcolormesh(M, cmap=SEQ, vmin=0, vmax=1, edgecolors=SURF, linewidth=2)
    annotate_heatmap(ax, M)
    ax.set_xticks(np.arange(len(X_ORDER)) + 0.5, X_ORDER)
    ax.set_yticks(np.arange(len(Y_ORDER)) + 0.5, Y_ORDER)
    ax.invert_yaxis()
    ax.grid(False)
    ax.set_title(r, fontsize=9.5)
    for s in ax.spines.values():
        s.set_visible(False)
for ax in axes.flat[len(RUN_ORDER) :]:
    ax.axis("off")
fig.suptitle(
    "Exactitud de posición conjunta por celda verdadera", x=0.01, ha="left", fontweight="semibold"
)
fig.tight_layout()
savefig(fig, "07_posicion_grilla")
plt.show()

# Sesgo marginal: ¿qué etiqueta de posición predice cada corrida?
pos_bias = pd.concat(
    {
        ax_: df.groupby("run", sort=False)[f"{ax_}_pred"]
        .value_counts(normalize=True, dropna=False)
        .unstack(fill_value=0)
        .reindex(RUN_ORDER)
        for ax_ in ["xpos", "ypos"]
    },
    axis=1,
)
pos_bias.columns = [f"{a}={v if isinstance(v, str) else '∅'}" for a, v in pos_bias.columns]
display(pos_bias.style.format("{:.2f}").background_gradient(cmap=SEQ, vmin=0, vmax=1))

### 9.3 Color del objeto en función del tono verdadero

Exactitud (tolerancia ±1 categoría) en ventanas de $15°$ de $h^{\text{obj}}$. La banda inferior muestra el tono de cada ventana.
Caídas localizadas indican regiones del círculo cromático que el modelo nombra de forma inconsistente.

In [ ]:
hbins = np.linspace(0, 1, 25)
hcent = (hbins[:-1] + hbins[1:]) / 2
df["hue_bin"] = pd.cut(df["object_color"], hbins, labels=hcent, include_lowest=True).astype(float)
fig, axes = plt.subplots(
    1, len(PROMPT_ORDER), figsize=(5.2 * len(PROMPT_ORDER), 3.8), sharey=True, squeeze=False
)
for ax, p in zip(axes[0], PROMPT_ORDER):
    for m in MODEL_ORDER:
        g = df[(df["prompt_label"] == p) & (df["model_label"] == m)]
        if g.empty:
            continue
        s = g.groupby("hue_bin")["color_obj_adj_correct"].mean().reindex(hcent)
        ax.plot(hcent * 360, s.values, "-o", ms=4, color=MODEL_COLORS[m], label=m)
    for c0, c1 in zip(hbins[:-1], hbins[1:]):
        ax.axvspan(
            c0 * 360,
            c1 * 360,
            ymin=0,
            ymax=0.035,
            color=colorsys.hsv_to_rgb((c0 + c1) / 2, 0.75, 0.9),
            lw=0,
        )
    ax.set(xlim=(0, 360), ylim=(-0.04, 1.02), xlabel="tono del objeto (grados)", title=p)
    ax.set_xticks(range(0, 361, 60))
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
axes[0, 0].set_ylabel("exactitud color objeto (±1)")
axes[0, -1].legend(loc="lower right")
fig.tight_layout()
savefig(fig, "08_color_vs_tono")
plt.show()

In [ ]:
# Confusión de categorías de color del objeto, agregada por modelo
CCOLS = COLOR_CATS + ["achromatic", "∅"]
fig, axes = plt.subplots(1, len(MODEL_ORDER), figsize=(4.6 * len(MODEL_ORDER), 4.2), squeeze=False)
for ax, m in zip(axes[0], MODEL_ORDER):
    g = df[df["model_label"] == m]
    ct = pd.crosstab(g["color_obj_true"], g["color_obj_pred"].fillna("∅")).reindex(
        index=COLOR_CATS, columns=CCOLS, fill_value=0
    )
    M = (ct.T / ct.sum(axis=1).replace(0, np.nan)).T.values
    ax.pcolormesh(M, cmap=SEQ, vmin=0, vmax=1, edgecolors=SURF, linewidth=1.5)
    annotate_heatmap(ax, M, fmt="{:.2f}")
    ax.set_xticks(np.arange(len(CCOLS)) + 0.5, CCOLS, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(np.arange(len(COLOR_CATS)) + 0.5, COLOR_CATS, fontsize=8)
    ax.invert_yaxis()
    ax.grid(False)
    ax.set_title(f"{m} (todos los prompts)", fontsize=9.5)
    for s in ax.spines.values():
        s.set_visible(False)
fig.supxlabel("color predicho (objeto)")
fig.supylabel("categoría verdadera")
fig.tight_layout()
savefig(fig, "09_confusion_color_objeto")
plt.show()

### 9.4 Contraste objeto–fondo

Hipótesis: cuando $|\Delta h| = |h^{\text{obj}} \ominus h^{\text{bg}}|$ es pequeño (objeto y fondo de tono similar), el modelo confunde
el color del objeto con el del fondo y reconoce peor la forma. Se grafica la exactitud en ventanas de $15°$ de $|\Delta h| \in [0°, 180°]$,
y se cuantifica la asociación con una regresión logística univariada por corrida,
$\operatorname{logit} P(\text{acierto}) = \beta_0 + \beta_1\,|\Delta h|_{[\text{rad}]}$ (Newton–Raphson; se reporta $\hat\beta_1$ y su valor-$p$ de Wald).

In [ ]:
df["contrast_deg"] = np.abs(360 * circ_diff(df["object_color"], df["background_color"]))
cbins = np.arange(0, 181, 15)
df["contrast_bin"] = (
    pd.cut(df["contrast_deg"], cbins, include_lowest=True).map(lambda iv: iv.mid).astype(float)
)


def logit_1d(x, y, iters=50):
    X = np.column_stack([np.ones_like(x), x])
    b = np.zeros(2)
    for _ in range(iters):
        p = 1 / (1 + np.exp(-X @ b))
        W = p * (1 - p)
        H = X.T @ (X * W[:, None])
        g = X.T @ (y - p)
        try:
            step = np.linalg.solve(H, g)
        except np.linalg.LinAlgError:
            return np.nan, np.nan
        b += step
        if np.abs(step).max() < 1e-8:
            break
    se = np.sqrt(np.diag(np.linalg.inv(H)))
    return b[1], 2 * stats.norm.sf(abs(b[1] / se[1]))


fig, axes = plt.subplots(1, 2, figsize=(12, 3.9), sharey=True)
lr_rows = []
for ax, target, title in zip(
    axes, ["color_obj_correct", "shape_correct"], ["Color del objeto (exacto)", "Forma"]
):
    for m in MODEL_ORDER:
        g = df[df["model_label"] == m]
        s = g.groupby("contrast_bin")[target].mean()
        ax.plot(s.index, s.values, "-o", ms=4, color=MODEL_COLORS[m], label=m)
    ax.set(xlabel="|Δ tono| objeto–fondo (grados)", title=title, xlim=(0, 180))
    ax.set_xticks(range(0, 181, 30))
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
    for r in RUN_ORDER:
        g = df[df["run"] == r]
        b1, p = logit_1d(np.deg2rad(g["contrast_deg"].values), g[target].astype(float).values)
        lr_rows.append(dict(run=r, atributo=target.replace("_correct", ""), beta1=b1, p_valor=p))
axes[0].set_ylabel("exactitud (todos los prompts)")
axes[1].legend(loc="lower right")
fig.tight_layout()
savefig(fig, "10_contraste_objeto_fondo")
plt.show()

lr_tab = (
    pd.DataFrame(lr_rows)
    .pivot(index="run", columns="atributo", values=["beta1", "p_valor"])
    .reindex(RUN_ORDER)
)
lr_tab.columns = [f"{a}_{b}" for a, b in lr_tab.columns]
display(lr_tab.style.format("{:.3g}"))
save_table(lr_tab, "contraste_logit")

## 10. Comparaciones pareadas

Todas las corridas describen **las mismas imágenes**, por lo que las diferencias de exactitud deben evaluarse de forma pareada.
Para dos corridas $A$ y $B$ sobre un atributo, sea $b$ el número de imágenes que $A$ acierta y $B$ no, y $c$ el recíproco.
Bajo $H_0: P(A \text{ acierta}, B \text{ falla}) = P(A \text{ falla}, B \text{ acierta})$, $b \mid (b+c) \sim \operatorname{Bin}(b+c, \tfrac12)$
(**McNemar exacto**). Los valores-$p$ se ajustan por **Holm–Bonferroni** dentro de cada familia (comparaciones entre prompts dentro de un modelo;
comparaciones entre modelos dentro de un prompt), controlando el FWER a nivel $\alpha$.

In [ ]:
def holm(p):
    p = np.asarray(p, float)
    order = np.argsort(p)
    m = len(p)
    adj = np.empty(m)
    running = 0.0
    for rank, idx in enumerate(order):
        running = max(running, (m - rank) * p[idx])
        adj[idx] = min(1.0, running)
    return adj


def mcnemar_pairs(
    family_col, compare_col, attrs=("shape", "pos_joint", "color_obj", "color_obj_adj", "core")
):
    rows = []
    for fam, g in df.groupby(family_col):
        levels = [
            l
            for l in (PROMPT_ORDER if compare_col == "prompt_label" else MODEL_ORDER)
            if l in set(g[compare_col])
        ]
        for a in attrs:
            W = g.pivot_table(
                index="_key", columns=compare_col, values=f"{a}_correct", aggfunc="first"
            )
            for A, B in itertools.combinations(levels, 2):
                pair = W[[A, B]].dropna().astype(bool)
                b = int((pair[A] & ~pair[B]).sum())
                c = int((~pair[A] & pair[B]).sum())
                p = stats.binomtest(b, b + c, 0.5).pvalue if b + c > 0 else 1.0
                rows.append(
                    {
                        family_col: fam,
                        "atributo": ATTR_LABELS[a],
                        "A": A,
                        "B": B,
                        "n": len(pair),
                        "acc_A": pair[A].mean(),
                        "acc_B": pair[B].mean(),
                        "delta": pair[A].mean() - pair[B].mean(),
                        "b": b,
                        "c": c,
                        "p": p,
                    }
                )
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    out["p_holm"] = out.groupby(family_col, group_keys=False)["p"].apply(
        lambda s: pd.Series(holm(s.values), index=s.index)
    )
    out["signif"] = out["p_holm"] < ALPHA
    return out


fmt = {"acc_A": "{:.3f}", "acc_B": "{:.3f}", "delta": "{:+.3f}", "p": "{:.2e}", "p_holm": "{:.2e}"}
if len(PROMPT_ORDER) > 1:
    mc_prompts = mcnemar_pairs("model_label", "prompt_label")
    display(Markdown("**Prompts dentro de cada modelo**"))
    display(
        mc_prompts.style.format(fmt).apply(
            lambda r: ["font-weight: bold" if r["signif"] else ""] * len(r), axis=1
        )
    )
    save_table(mc_prompts, "mcnemar_prompts", index=False)
if len(MODEL_ORDER) > 1:
    mc_models = mcnemar_pairs("prompt_label", "model_label")
    display(Markdown("**Modelos dentro de cada prompt**"))
    display(
        mc_models.style.format(fmt).apply(
            lambda r: ["font-weight: bold" if r["signif"] else ""] * len(r), axis=1
        )
    )
    save_table(mc_models, "mcnemar_modelos", index=False)

In [ ]:
# Resumen visual: Δ exactitud (fila − columna) entre modelos, por prompt y atributo
if len(MODEL_ORDER) > 1:
    show_attrs = ["Forma", "Posición (x,y)", "Color objeto ±1"]
    fig, axes = plt.subplots(
        len(PROMPT_ORDER),
        len(show_attrs),
        figsize=(3.4 * len(show_attrs) + 1, 3.0 * len(PROMPT_ORDER)),
        squeeze=False,
    )
    lim = max(0.05, mc_models["delta"].abs().max())
    for i, p in enumerate(PROMPT_ORDER):
        for j, a in enumerate(show_attrs):
            ax = axes[i, j]
            sub = mc_models[(mc_models["prompt_label"] == p) & (mc_models["atributo"] == a)]
            M = pd.DataFrame(np.nan, index=MODEL_ORDER, columns=MODEL_ORDER)
            S = pd.DataFrame(False, index=MODEL_ORDER, columns=MODEL_ORDER)
            for _, rr in sub.iterrows():
                M.at[rr["A"], rr["B"]], M.at[rr["B"], rr["A"]] = rr["delta"], -rr["delta"]
                S.at[rr["A"], rr["B"]] = S.at[rr["B"], rr["A"]] = rr["signif"]
            ax.pcolormesh(M.values, cmap=DIV, vmin=-lim, vmax=lim, edgecolors=SURF, linewidth=2)
            for (rr_, cc_), v in np.ndenumerate(M.values):
                if not np.isnan(v):
                    ax.text(
                        cc_ + 0.5,
                        rr_ + 0.5,
                        f"{v:+.2f}" + ("*" if S.values[rr_, cc_] else ""),
                        ha="center",
                        va="center",
                        fontsize=8,
                        color="white" if abs(v) > 0.6 * lim else INK,
                    )
            ax.set_xticks(
                np.arange(len(MODEL_ORDER)) + 0.5, MODEL_ORDER, rotation=30, ha="right", fontsize=8
            )
            ax.set_yticks(np.arange(len(MODEL_ORDER)) + 0.5, MODEL_ORDER, fontsize=8)
            ax.invert_yaxis()
            ax.grid(False)
            ax.set_title(f"{a} · {p}", fontsize=9)
            for s in ax.spines.values():
                s.set_visible(False)
    fig.suptitle(
        "Δ exactitud (fila − columna); * = significativo tras Holm",
        x=0.01,
        ha="left",
        fontweight="semibold",
    )
    fig.tight_layout()
    savefig(fig, "11_mcnemar_modelos")
    plt.show()

## 11. Dificultad por imagen y ejemplos cualitativos

La **dificultad** de la imagen $i$ para un atributo es $d_i = 1 - \frac{1}{R}\sum_r \mathbb 1[\hat z_{i,r} = z_i]$ sobre las $R$ corridas.
Si los errores fueran independientes entre corridas con tasa común, $d_i$ se concentraría en torno a su media; una distribución bimodal
(muchas imágenes que *todas* las corridas fallan) indica dificultad intrínseca de ciertas configuraciones de factores.

In [ ]:
diff = (
    df.groupby("_key")[
        ["shape_correct", "pos_joint_correct", "color_obj_adj_correct", "core_correct"]
    ]
    .mean()
    .rsub(1)
    .add_prefix("dif_")
    .join(manifest.set_index("_key")[["image_id"] + ATTR_COLS])
)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4), sharey=True)
R = len(RUN_ORDER)
edges = np.linspace(-0.5 / R, 1 + 0.5 / R, R + 2)
for ax, c, t in zip(
    axes,
    ["dif_shape_correct", "dif_pos_joint_correct", "dif_color_obj_adj_correct"],
    ["Forma", "Posición (x,y)", "Color objeto ±1"],
):
    ax.hist(diff[c], bins=edges, color=SERIES[0], edgecolor=SURF, linewidth=1.5)
    ax.set(title=t, xlabel="fracción de corridas que fallan")
    ax.grid(axis="x", visible=False)
axes[0].set_ylabel("imágenes")
fig.tight_layout()
savefig(fig, "12_dificultad_por_imagen")
plt.show()

diff["shape_name"] = diff["object_shape"].map(SHAPE_NAMES)
by_shape = diff.groupby("shape_name")[
    ["dif_shape_correct", "dif_pos_joint_correct", "dif_color_obj_adj_correct"]
].mean()
display(
    by_shape.style.format("{:.3f}")
    .background_gradient(cmap=SEQ, vmin=0, vmax=1)
    .set_caption("Dificultad media por forma verdadera")
)
save_table(by_shape, "dificultad_por_forma")

In [ ]:
def show_examples(frame, n=N_EXAMPLES, title=None, seed=SEED):
    cols = [
        "run",
        "image_id",
        "prediction",
        "reference",
        "shape_true",
        "shape_pred",
        "xpos_true",
        "xpos_pred",
        "ypos_true",
        "ypos_pred",
        "color_obj_true",
        "color_obj_pred",
        "color_bg_true",
        "color_bg_pred",
    ]
    if title:
        display(Markdown(f"**{title}**"))
    if frame.empty:
        print("(sin casos)")
        return
    display(frame.sample(min(n, len(frame)), random_state=seed)[cols].reset_index(drop=True))


# Muestra aleatoria por corrida
for r in RUN_ORDER:
    show_examples(df[df["run"] == r], title=f"Muestra aleatoria · {r}")

In [ ]:
# Casos extremos
show_examples(
    df[df["n_words"] > 0].nlargest(200, "rep_trigram"),
    n=4,
    title="Mayor repetición (posibles bucles)",
)
show_examples(df[~df["ends_clean"] & (df["n_words"] > 0)], n=4, title="Probablemente truncadas")
hard = diff[diff["dif_core_correct"] == 1].index
show_examples(
    df[df["_key"].isin(hard) & ~df["shape_correct"] & ~df["color_obj_adj_correct"]],
    n=6,
    title="Imágenes que ninguna corrida describe bien (forma y color errados)",
)

## 12. Exportación

Se guarda la tabla por fila con todos los atributos extraídos (para métricas posteriores y para auditar los extractores) y el resumen
compacto por corrida. Todas las tablas quedan en `reports/tables/eval/` (CSV y `.tex`) y las figuras en `reports/figures/eval/` (PNG y PDF).

In [ ]:
keep_cols = (
    [
        "exp_id",
        "exp",
        "dataset",
        "split",
        "model",
        "model_label",
        "prompt",
        "prompt_id",
        "prompt_name",
        "run",
        "image_id",
        "prediction",
        "reference",
    ]
    + ATTR_COLS
    + [c for a in ATTRS for c in (f"{a}_true", f"{a}_pred", f"{a}_mentioned", f"{a}_correct")]
    + [
        "color_obj_adj_correct",
        "color_bg_adj_correct",
        "color_spot_adj_correct",
        "pos_joint_correct",
        "core_correct",
        "shape_halluc",
        "n_words",
        "n_chars",
        "n_sents",
        "rep_trigram",
        "ends_clean",
        "unigram_f1",
        "contrast_deg",
    ]
)
per_row = df[keep_cols].copy()
per_row.to_parquet(TAB_DIR / "predicciones_atributos.parquet", index=False)

summary = (
    acc.pivot(index="run", columns="attr", values="exactitud")[
        ["shape", "pos_joint", "color_obj", "color_obj_adj", "core"]
    ]
    .rename(columns=ATTR_LABELS)
    .join(length_tab[["palabras_media", "truncadas"]])
    .join(vocab_tab[["distinct_2"]])
    .reindex(RUN_ORDER)
)
display(
    summary.style.format("{:.3f}").background_gradient(
        cmap=SEQ,
        vmin=0,
        vmax=1,
        subset=[
            ATTR_LABELS[a] for a in ["shape", "pos_joint", "color_obj", "color_obj_adj", "core"]
        ],
    )
)
save_table(summary, "resumen_eda")
print("Figuras:", sorted(p.name for p in FIG_DIR.glob("*.png")))
print("Tablas :", sorted(p.name for p in TAB_DIR.glob("*.csv")))

## Notas metodológicas y limitaciones

1. **Extracción léxica.** Las tasas dependen de los léxicos de la sección 4. Revise la tabla de términos con forma no detectada (§7) y amplíe
   `SHAPE_LEXICON`/`COLOR_LEXICON` si aparecen sinónimos legítimos; errores genuinos (*dragon → dinosaur*) no deben agregarse.
2. **Asignación de color por contexto.** Es una heurística de ventana; frases como *"a dragon in front of a blue wall, painted red"* pueden asignarse mal.
   La tasa de mención del color de fondo/foco depende más del prompt que del modelo.
3. **Discretización del tono.** Los bordes entre categorías son arbitrarios; por eso se reporta también la tolerancia ±1. La validación de §4.4
   indica cuánto se aleja el propio generador de texto de M3Di de esta discretización (cota de lo alcanzable).
4. **Posición.** Menciones de *left/right/top* que no refieren a la ubicación del objeto (p. ej. *"light coming from the left"*) producen
   falsos positivos; las etiquetas `ambiguous` (menciona ambos extremos de un eje) se cuentan como error.
5. **Inferencia.** Las comparaciones usan McNemar exacto porque las corridas son pareadas por imagen; no se deben comparar IC de Wilson
   solapados como si fueran pruebas independientes.